# LNPHub oracle subset extraction: in vitro expression data

This notebook extracts the first regression-oracle subset from `lnphub_master_v2.0.0.csv`:

- `experimental_setting == "in_vitro"`
- `delivery_metric == "expression"`
- excludes `expression_discretized`
- keeps only studies with more than 400 usable rows
- saves the subset without modifying the master file

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

CSV_PATH = Path("/content/lnphub_master_v2.0.0.csv")
OUTDIR = Path("/content/notebooks/oracle_subsets")
OUTDIR.mkdir(parents=True, exist_ok=True)

MIN_ROWS_PER_STUDY = 400
SETTING = "in_vitro"
DELIVERY_METRIC = "expression"

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH}. Upload lnphub_master_v2.0.0.csv to /content first."
    )

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded: {CSV_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

In [ ]:
required = ["study_id", "experimental_setting", "delivery_metric", "delivery"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

delivery_numeric = pd.to_numeric(df["delivery"], errors="coerce")
mask = (
    df["experimental_setting"].astype(str).str.strip().str.lower().eq(SETTING)
    & df["delivery_metric"].astype(str).str.strip().str.lower().eq(DELIVERY_METRIC)
    & delivery_numeric.notna()
)

candidate = df.loc[mask].copy()
candidate["delivery"] = pd.to_numeric(candidate["delivery"], errors="coerce")

study_summary = (
    candidate.groupby("study_id", dropna=False)
    .agg(
        n_rows=("study_id", "size"),
        cargo_types=("cargo_type", lambda s: "; ".join(sorted(s.dropna().astype(str).unique())) if "cargo_type" in candidate else ""),
        biological_models=("biological_model", lambda s: "; ".join(sorted(s.dropna().astype(str).unique())[:5]) if "biological_model" in candidate else ""),
        delivery_min=("delivery", "min"),
        delivery_max=("delivery", "max"),
        delivery_mean=("delivery", "mean"),
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

eligible_studies = study_summary.loc[study_summary["n_rows"] > MIN_ROWS_PER_STUDY, "study_id"]
subset = candidate[candidate["study_id"].isin(eligible_studies)].copy()

eligible_summary = study_summary[study_summary["study_id"].isin(eligible_studies)].reset_index(drop=True)

display(eligible_summary)
print(f"Candidate in-vitro expression rows: {len(candidate):,}")
print(f"Eligible studies > {MIN_ROWS_PER_STUDY} rows: {eligible_studies.nunique():,}")
print(f"Final subset rows: {len(subset):,}")

In [ ]:
assert len(subset) > MIN_ROWS_PER_STUDY
assert subset["experimental_setting"].astype(str).str.lower().eq(SETTING).all()
assert subset["delivery_metric"].astype(str).str.lower().eq(DELIVERY_METRIC).all()
assert not subset["delivery_metric"].astype(str).str.contains("discretized", case=False, na=False).any()
assert pd.to_numeric(subset["delivery"], errors="coerce").notna().all()

qc = pd.DataFrame([{
    "subset": "in_vitro_expression_gt400",
    "n_rows": len(subset),
    "n_studies": subset["study_id"].nunique(),
    "n_unique_lnps": subset["lnp_id"].nunique() if "lnp_id" in subset.columns else np.nan,
    "n_unique_ionizable_lipids": subset["il_smiles"].nunique() if "il_smiles" in subset.columns else np.nan,
    "delivery_metric": DELIVERY_METRIC,
    "experimental_setting": SETTING,
    "min_rows_per_study": MIN_ROWS_PER_STUDY,
}])

display(qc)

In [ ]:
subset_path = OUTDIR / "lnphub_oracle_in_vitro_expression_gt400.csv"
summary_path = OUTDIR / "lnphub_oracle_in_vitro_expression_gt400_study_summary.csv"
qc_path = OUTDIR / "lnphub_oracle_in_vitro_expression_gt400_qc.csv"

subset.to_csv(subset_path, index=False)
eligible_summary.to_csv(summary_path, index=False)
qc.to_csv(qc_path, index=False)

print(f"Saved subset  -> {subset_path}")
print(f"Saved summary -> {summary_path}")
print(f"Saved QC      -> {qc_path}")

## Encode ionizable-lipid SMILES with pretrained molecular encoders

Outputs are written to `/content/notebooks/encoded_in_vitro/`. Each encoder gets one vector file containing `record_id`, LNP/IL identifiers, the original/canonical SMILES, and embedding columns.

Direct Colab encoders included here:

- ChemBERTa-MTR
- MoLFormer
- Uni-Mol

Repo/checkpoint-dependent encoders checked here:

- AGILE
- KPGT
- LiON
- MolCLR-GIN

Those four require their original repository checkpoints and environment-specific code. The notebook prints exactly what is present/missing so we do not accidentally create fake baseline vectors.

In [ ]:
# Encoder dependencies
!pip -q install pandas numpy tqdm pyarrow transformers accelerate safetensors rdkit-pypi unimol-tools cairosvg

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

SUBSET_PATH = Path("/content/notebooks/oracle_subsets/lnphub_oracle_in_vitro_expression_gt400.csv")
ENCODED_DIR = Path("/content/notebooks/encoded_in_vitro")
ENCODED_DIR.mkdir(parents=True, exist_ok=True)

if not SUBSET_PATH.exists():
    raise FileNotFoundError(
        f"Subset not found: {SUBSET_PATH}. Run the extraction cells above first."
    )

subset = pd.read_csv(SUBSET_PATH, low_memory=False)
SMILES_COL = "il_smiles"
if SMILES_COL not in subset.columns:
    raise ValueError(f"Missing SMILES column: {SMILES_COL}")

id_cols = [c for c in ["record_id", "study_id", "lnp_id", "il_id", "il_smiles", "delivery"] if c in subset.columns]
print(f"Loaded subset: {len(subset):,} rows")
print(f"Output folder: {ENCODED_DIR}")

In [ ]:
# Canonicalize unique ionizable-lipid SMILES once, then merge vectors back to records.
from rdkit import Chem

ids = subset[id_cols].copy()
ids[SMILES_COL] = ids[SMILES_COL].astype(str).str.strip()

def canonicalize_smiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else np.nan

print("Canonicalizing SMILES...")
ids["canonical_il_smiles"] = [canonicalize_smiles(s) for s in tqdm(ids[SMILES_COL], desc="RDKit")]
valid_ids = ids.dropna(subset=["canonical_il_smiles"]).copy()
unique_smiles = valid_ids[["canonical_il_smiles"]].drop_duplicates().reset_index(drop=True)

print(f"Valid rows for encoding: {len(valid_ids):,} / {len(ids):,}")
print(f"Unique IL SMILES to encode: {len(unique_smiles):,}")
display(unique_smiles.head())

In [ ]:
# Common save helpers

def save_encoder_vectors(encoder_name, vectors, feature_prefix):
    vectors = np.asarray(vectors, dtype=np.float32)
    if vectors.shape[0] != len(unique_smiles):
        raise ValueError(f"{encoder_name}: vectors rows {vectors.shape[0]} != unique SMILES {len(unique_smiles)}")

    vec_cols = [f"{feature_prefix}_{i:04d}" for i in range(vectors.shape[1])]
    unique_vec = pd.concat(
        [unique_smiles.reset_index(drop=True), pd.DataFrame(vectors, columns=vec_cols)],
        axis=1,
    )

    record_vec = ids.merge(unique_vec, on="canonical_il_smiles", how="left")
    base = encoder_name.replace(" ", "_").replace("-", "_") + "_encoder_vectors"
    out_path = ENCODED_DIR / f"{base}.parquet"
    record_vec.to_parquet(out_path, index=False)

    manifest = {
        "encoder": encoder_name,
        "file": str(out_path),
        "n_records": int(len(record_vec)),
        "n_valid_records": int(record_vec["canonical_il_smiles"].notna().sum()),
        "n_unique_smiles": int(len(unique_smiles)),
        "embedding_dim": int(vectors.shape[1]),
        "id_columns": id_cols + ["canonical_il_smiles"],
    }
    with open(ENCODED_DIR / f"{base}_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"Saved {encoder_name}: {out_path}")
    print(f"Shape: {record_vec.shape[0]:,} rows × {record_vec.shape[1]:,} columns")
    return out_path

encoder_status = []

def mark_status(encoder, status, message, output_file=None):
    row = {"encoder": encoder, "status": status, "message": message, "output_file": str(output_file) if output_file else ""}
    encoder_status.append(row)
    print(f"[{encoder}] {status}: {message}")
    pd.DataFrame(encoder_status).to_csv(ENCODED_DIR / "encoder_status.csv", index=False)

In [ ]:
# ChemBERTa-MTR and MoLFormer encoders from Hugging Face
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch device: {DEVICE}")

@torch.inference_mode()
def encode_hf_smiles(model_id, encoder_name, prefix, batch_size=32, max_length=256, trust_remote_code=True):
    print(f"
=== {encoder_name} ===")
    print(f"Loading tokenizer/model: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    try:
        model = AutoModel.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    except Exception as e:
        print(f"AutoModel failed, trying AutoModelForSequenceClassification: {e}")
        model = AutoModelForSequenceClassification.from_pretrained(model_id, trust_remote_code=trust_remote_code)

    model.to(DEVICE)
    model.eval()

    smiles = unique_smiles["canonical_il_smiles"].astype(str).tolist()
    chunks = []
    for start in tqdm(range(0, len(smiles), batch_size), desc=f"Encoding {encoder_name}"):
        batch = smiles[start:start + batch_size]
        tok = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        out = model(**tok, output_hidden_states=True, return_dict=True)

        hidden = getattr(out, "last_hidden_state", None)
        if hidden is None:
            hidden = out.hidden_states[-1]

        mask = tok.get("attention_mask", torch.ones(hidden.shape[:2], device=hidden.device)).unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        chunks.append(pooled.detach().cpu().numpy().astype(np.float32))

    vectors = np.vstack(chunks)
    return save_encoder_vectors(encoder_name, vectors, prefix)

hf_jobs = [
    ("DeepChem/ChemBERTa-77M-MTR", "ChemBERTa_MTR", "chemberta_mtr", 32, 256, False),
    ("ibm-research/MoLFormer-XL-both-10pct", "MoLFormer", "molformer", 16, 256, True),
]

for model_id, encoder_name, prefix, batch_size, max_length, trust_remote_code in hf_jobs:
    try:
        out = encode_hf_smiles(model_id, encoder_name, prefix, batch_size, max_length, trust_remote_code)
        mark_status(encoder_name, "encoded", "completed", out)
    except Exception as e:
        mark_status(encoder_name, "failed", repr(e))

In [ ]:
# Uni-Mol encoder
# Uses unimol-tools molecular representation interface.
import numpy as np
from tqdm.auto import tqdm

try:
    from unimol_tools import UniMolRepr

    def extract_unimol_cls_repr(rep):
        if isinstance(rep, dict):
            if "cls_repr" in rep:
                return np.asarray(rep["cls_repr"], dtype=np.float32)
            if "molecular_repr" in rep:
                return np.asarray(rep["molecular_repr"], dtype=np.float32)

        if isinstance(rep, list):
            if len(rep) and isinstance(rep[0], dict):
                key = "cls_repr" if "cls_repr" in rep[0] else "molecular_repr"
                return np.vstack([np.asarray(x[key], dtype=np.float32) for x in rep])
            return np.asarray(rep, dtype=np.float32)

        arr = np.asarray(rep, dtype=np.float32)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        return arr

    print("\n=== Uni-Mol ===")
    smiles = unique_smiles["canonical_il_smiles"].astype(str).tolist()
    clf = UniMolRepr(data_type="molecule", remove_hs=False)

    chunks = []
    BATCH_SIZE = 64
    for start in tqdm(range(0, len(smiles), BATCH_SIZE), desc="Encoding Uni-Mol"):
        batch = smiles[start:start + BATCH_SIZE]
        rep = clf.get_repr(batch, return_atomic_reprs=False)
        chunks.append(extract_unimol_cls_repr(rep))

    vectors = np.vstack(chunks).astype(np.float32)
    out = save_encoder_vectors("Uni_Mol", vectors, "unimol")
    mark_status("Uni_Mol", "encoded", "completed", out)
except Exception as e:
    mark_status("Uni_Mol", "failed", repr(e))


In [ ]:
# Access checks for repo/checkpoint-dependent baselines.
# This cell does NOT create fake vectors. It verifies whether the original code/checkpoints are available.
import subprocess
from pathlib import Path
import pandas as pd

REPO_DIR = Path("/content/notebooks/encoder_repos")
REPO_DIR.mkdir(parents=True, exist_ok=True)

def clone_if_needed(name, url):
    dest = REPO_DIR / name
    if dest.exists():
        print(f"{name}: repo already present -> {dest}")
        return dest
    print(f"{name}: cloning {url}")
    subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=False)
    return dest

checks = []

agile = clone_if_needed("AGILE", "https://github.com/bowang-lab/AGILE.git")
checks.append({
    "encoder": "AGILE",
    "repo": str(agile),
    "checkpoint_expected": str(agile / "ckpt" / "pretrained_agile_60k"),
    "checkpoint_present": (agile / "ckpt" / "pretrained_agile_60k").exists(),
    "encoding_status": "needs original AGILE checkpoint/inference adapter before vectors can be generated",
})

kpgt = clone_if_needed("KPGT", "https://github.com/lihan97/kpgt.git")
checks.append({
    "encoder": "KPGT",
    "repo": str(kpgt),
    "checkpoint_expected": str(kpgt / "models"),
    "checkpoint_present": (kpgt / "models").exists(),
    "encoding_status": "needs KPGT pretrained model zip placed under KPGT/models before vectors can be generated",
})

molclr = clone_if_needed("MolCLR", "https://github.com/yuyangw/MolCLR.git")
checks.append({
    "encoder": "MolCLR_GIN",
    "repo": str(molclr),
    "checkpoint_expected": str(molclr / "ckpt" / "pretrained_gin"),
    "checkpoint_present": (molclr / "ckpt" / "pretrained_gin").exists(),
    "encoding_status": "needs MolCLR pretrained GIN checkpoint and compatible PyG environment before vectors can be generated",
})

lnpdb = clone_if_needed("LNPDB", "https://github.com/evancollins1/LNPDB.git")
checks.append({
    "encoder": "LiON",
    "repo": str(lnpdb),
    "checkpoint_expected": str(lnpdb / "data" / "LNPDB_for_LiON"),
    "checkpoint_present": (lnpdb / "data" / "LNPDB_for_LiON").exists(),
    "encoding_status": "LiON is formulation/checkpoint-specific; use after matching required LiON input schema, not just il_smiles",
})

repo_check_table = pd.DataFrame(checks)
repo_check_table.to_csv(ENCODED_DIR / "repo_based_encoder_access_check.csv", index=False)
display(repo_check_table)

for _, row in repo_check_table.iterrows():
    mark_status(row["encoder"], "checked", row["encoding_status"])

In [ ]:
# Final status summary
status = pd.read_csv(ENCODED_DIR / "encoder_status.csv")
display(status)
print(f"All outputs are under: {ENCODED_DIR}")
print("Generated vector files:")
for p in sorted(ENCODED_DIR.glob("*_encoder_vectors.parquet")):
    print(" -", p)

## Actual repo-based baseline encoders: AGILE, MolCLR-GIN, KPGT, LiON

These cells generate real baseline embeddings when the required official checkpoints are available. AGILE and MolCLR-GIN use the pretrained checkpoints included in their GitHub repositories. KPGT uses the official `extract_features.py` path and needs `models/pretrained/base/base.pth`. LiON uses the LNPDB chemprop checkpoint and extracts fingerprints from the penultimate layer.

In [ ]:
# Repo-based encoder dependencies
# AGILE + MolCLR-GIN run in the current Colab runtime.
# KPGT needs DGL/DGLLife plus its official base.pth checkpoint.
# LiON needs Chemprop 1.7.0, so its cell creates a Python 3.8 env separately if needed.

import sys, subprocess, shutil, os
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

def run(cmd, check=False):
    print("$", " ".join(map(str, cmd)))
    out = subprocess.run(cmd, text=True, capture_output=True)
    print(out.stdout[-2500:])
    if out.stderr:
        print(out.stderr[-2500:])
    if check and out.returncode != 0:
        raise RuntimeError(f"Command failed with code {out.returncode}: {' '.join(map(str, cmd))}")
    return out.returncode

run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric", "rdkit-pypi", "pyarrow", "tqdm", "pyyaml"])
run([sys.executable, "-m", "pip", "install", "-q", "dgl", "dgllife"])

print("Dependency cell finished. If Colab asks for a runtime restart after package installs, restart and rerun from the setup cells.")

In [ ]:
# Clone official repos and verify bundled checkpoints
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display

REPO_DIR = Path("/content/notebooks/encoder_repos")
REPO_DIR.mkdir(parents=True, exist_ok=True)

def clone_if_needed(name, url):
    dest = REPO_DIR / name
    if dest.exists():
        print(f"{name}: repo already present -> {dest}")
    else:
        print(f"{name}: cloning {url}")
        subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)
    return dest

AGILE_DIR = clone_if_needed("AGILE", "https://github.com/bowang-lab/AGILE.git")
MOLCLR_DIR = clone_if_needed("MolCLR", "https://github.com/yuyangw/MolCLR.git")
KPGT_DIR = clone_if_needed("KPGT", "https://github.com/lihan97/kpgt.git")
LNPDB_DIR = clone_if_needed("LNPDB", "https://github.com/evancollins1/LNPDB.git")

repo_paths = pd.DataFrame([
    {"encoder": "AGILE", "path": str(AGILE_DIR / "ckpt/pretrained_agile_60k/checkpoints/model.pth")},
    {"encoder": "MolCLR_GIN", "path": str(MOLCLR_DIR / "ckpt/pretrained_gin/checkpoints/model.pth")},
    {"encoder": "KPGT", "path": str(KPGT_DIR / "models/pretrained/base/base.pth")},
    {"encoder": "LiON", "path": str(LNPDB_DIR / "data/LNPDB_for_LiON/single_split/trained_model_checkpoints")},
])
repo_paths["exists"] = repo_paths["path"].map(lambda p: Path(p).exists())
repo_paths["size_MB"] = repo_paths["path"].map(lambda p: round(Path(p).stat().st_size / 1e6, 3) if Path(p).is_file() else np.nan)
display(repo_paths)

In [ ]:
# Actual AGILE and MolCLR-GIN encoding from official pretrained GIN-style checkpoints

import sys, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from rdkit import Chem
from torch_geometric.data import Data, Batch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

ATOM_LIST = list(range(1, 119))
CHIRALITY_LIST = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
]
BOND_LIST = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]
BONDDIR_LIST = [
    Chem.rdchem.BondDir.NONE,
    Chem.rdchem.BondDir.ENDUPRIGHT,
    Chem.rdchem.BondDir.ENDDOWNRIGHT,
]

def load_class_from_file(path, class_name, module_name):
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return getattr(mod, class_name)

def smiles_to_pyg_graph(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    mol = Chem.AddHs(mol)

    atoms = []
    for atom in mol.GetAtoms():
        atomic_num = atom.GetAtomicNum()
        if atomic_num not in ATOM_LIST:
            return None
        chiral = atom.GetChiralTag()
        chirality_idx = CHIRALITY_LIST.index(chiral) if chiral in CHIRALITY_LIST else 0
        atoms.append([ATOM_LIST.index(atomic_num), chirality_idx])

    row, col, edge_feat = [], [], []
    for bond in mol.GetBonds():
        bt, bd = bond.GetBondType(), bond.GetBondDir()
        btype = BOND_LIST.index(bt) if bt in BOND_LIST else 0
        bdir = BONDDIR_LIST.index(bd) if bd in BONDDIR_LIST else 0
        a, b = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        row += [a, b]
        col += [b, a]
        edge_feat += [[btype, bdir], [btype, bdir]]

    return Data(
        x=torch.tensor(atoms, dtype=torch.long),
        edge_index=torch.tensor([row, col], dtype=torch.long),
        edge_attr=torch.tensor(edge_feat, dtype=torch.long),
        y=torch.zeros((1, 1), dtype=torch.float),
    )

def load_state_dict_safely(path):
    obj = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(obj, dict) and "state_dict" in obj:
        obj = obj["state_dict"]
    if isinstance(obj, dict) and "model_state_dict" in obj:
        obj = obj["model_state_dict"]
    return {k.replace("module.", ""): v for k, v in obj.items()}

@torch.inference_mode()
def encode_gnn_checkpoint(encoder_name, model_class, ckpt_path, prefix, batch_size=128):
    print(f"\n=== {encoder_name} ===")
    ckpt_path = Path(ckpt_path)
    print("Checkpoint:", ckpt_path)

    if not ckpt_path.exists() or ckpt_path.stat().st_size < 100000:
        mark_status(encoder_name, "failed", f"missing or incomplete checkpoint: {ckpt_path}")
        return None

    model = model_class(num_layer=5, emb_dim=300, feat_dim=512, drop_ratio=0, pool="mean").to(DEVICE)
    msg = model.load_state_dict(load_state_dict_safely(ckpt_path), strict=False)
    print("Missing keys:", len(msg.missing_keys), "Unexpected keys:", len(msg.unexpected_keys))
    model.eval()

    smiles = unique_smiles["canonical_il_smiles"].astype(str).tolist()
    vectors = np.full((len(smiles), 512), np.nan, dtype=np.float32)
    failed, graph_items = [], []

    for i, smi in enumerate(tqdm(smiles, desc=f"Building graphs for {encoder_name}")):
        g = smiles_to_pyg_graph(smi)
        if g is None:
            failed.append({"row": i, "canonical_il_smiles": smi})
        else:
            graph_items.append((i, g))

    print(f"Graphs built: {len(graph_items):,}; failed: {len(failed):,}")
    if failed:
        pd.DataFrame(failed).to_csv(ENCODED_DIR / f"{prefix}_failed_smiles.csv", index=False)

    for start in tqdm(range(0, len(graph_items), batch_size), desc=f"Encoding {encoder_name}"):
        batch_items = graph_items[start:start + batch_size]
        idxs = [i for i, _ in batch_items]
        batch = Batch.from_data_list([g for _, g in batch_items]).to(DEVICE)
        h, _ = model(batch)
        vectors[idxs, :] = h.detach().cpu().numpy().astype(np.float32)

    out = save_encoder_vectors(encoder_name, vectors, prefix)
    mark_status(encoder_name, "encoded", f"completed; failed_graphs={len(failed)}", out)
    return out

AGILE = load_class_from_file(AGILE_DIR / "models" / "agile_pretrain.py", "AGILE", "agile_pretrain_local")
GINet = load_class_from_file(MOLCLR_DIR / "models" / "ginet_molclr.py", "GINet", "molclr_ginet_local")

encode_gnn_checkpoint(
    "AGILE",
    AGILE,
    AGILE_DIR / "ckpt" / "pretrained_agile_60k" / "checkpoints" / "model.pth",
    "agile",
)

encode_gnn_checkpoint(
    "MolCLR_GIN",
    GINet,
    MOLCLR_DIR / "ckpt" / "pretrained_gin" / "checkpoints" / "model.pth",
    "molclr_gin",
)

In [ ]:
# Actual KPGT encoding using official KPGT preprocessing + extract_features.py
# Required checkpoint: /content/notebooks/encoder_repos/KPGT/models/pretrained/base/base.pth

import shutil, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

print("\n=== KPGT ===")
kpgt_root = Path(KPGT_DIR)
kpgt_data_root = kpgt_root / "datasets"
kpgt_dataset = "lnphub_invitro_expression"
kpgt_dataset_dir = kpgt_data_root / kpgt_dataset
kpgt_dataset_dir.mkdir(parents=True, exist_ok=True)

kpgt_input = unique_smiles.rename(columns={"canonical_il_smiles": "smiles"}).copy()
kpgt_input["dummy"] = 0.0
kpgt_csv = kpgt_dataset_dir / f"{kpgt_dataset}.csv"
kpgt_input.to_csv(kpgt_csv, index=False)
print(f"KPGT input saved: {kpgt_csv}")

kpgt_model_path = kpgt_root / "models" / "pretrained" / "base" / "base.pth"
for candidate in [Path("/content/base.pth"), Path("/content/notebooks/base.pth"), ENCODED_DIR / "base.pth"]:
    if candidate.exists() and not kpgt_model_path.exists():
        kpgt_model_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(candidate, kpgt_model_path)
        print(f"Copied KPGT checkpoint from {candidate} -> {kpgt_model_path}")

if not kpgt_model_path.exists():
    mark_status(
        "KPGT",
        "failed",
        "missing checkpoint base.pth; place it at /content/notebooks/encoder_repos/KPGT/models/pretrained/base/base.pth",
    )
    display(pd.DataFrame([{
        "encoder": "KPGT",
        "status": "missing_checkpoint",
        "needed_file": str(kpgt_model_path),
        "note": "Download the official pretrained KPGT base.pth from the Figshare link in the KPGT README, then rerun this cell.",
    }]))
else:
    print("Preprocessing KPGT graphs, fingerprints, and descriptors...")
    subprocess.run(
        [sys.executable, "scripts/preprocess_downstream_dataset.py", "--data_path", str(kpgt_data_root), "--dataset", kpgt_dataset, "--n_jobs", "2"],
        cwd=str(kpgt_root),
        check=True,
    )

    print("Extracting KPGT latent features...")
    subprocess.run(
        [sys.executable, "extract_features.py", "--config", "base", "--model_path", str(kpgt_model_path), "--data_path", str(kpgt_data_root), "--dataset", kpgt_dataset],
        cwd=str(kpgt_root / "scripts"),
        check=True,
    )

    npz_path = kpgt_dataset_dir / "kpgt_base.npz"
    vectors = np.load(npz_path)["fps"].astype(np.float32)
    out = save_encoder_vectors("KPGT", vectors, "kpgt")
    mark_status("KPGT", "encoded", "completed", out)

In [ ]:
# Actual LiON fingerprint extraction using LNPDB Chemprop checkpoint
# LiON is formulation-level: it uses IL_SMILES plus formulation/context features aligned to LNPDB extra_x.
# If the current Colab Python is too new for chemprop==1.7.0, this cell creates a local Python 3.8 env using uv.

import os, sys, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

print("\n=== LiON ===")
lion_dir = Path(LNPDB_DIR) / "data" / "LNPDB_for_LiON"
lion_ckpt = lion_dir / "single_split" / "trained_model_checkpoints"
ref_extra = lion_dir / "all_data_extra_x.csv"

def shell(cmd, check=False):
    print("$", " ".join(map(str, cmd)))
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.stdout:
        print(res.stdout[-3500:])
    if res.stderr:
        print(res.stderr[-3500:])
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed with code {res.returncode}: {' '.join(map(str, cmd))}")
    return res

def find_chemprop_fingerprint():
    exe = shutil.which("chemprop_fingerprint")
    if exe:
        return exe
    venv_exe = Path("/content/notebooks/lion_py38/bin/chemprop_fingerprint")
    return str(venv_exe) if venv_exe.exists() else None

def ensure_lion_chemprop():
    exe = find_chemprop_fingerprint()
    if exe:
        print("Using chemprop_fingerprint:", exe)
        return exe

    print("chemprop_fingerprint not found. Creating Python 3.8 env with uv...")
    shell([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    shell(["uv", "python", "install", "3.8"], check=True)
    shell(["uv", "venv", "/content/notebooks/lion_py38", "--python", "3.8"], check=True)
    py38 = "/content/notebooks/lion_py38/bin/python"
    shell(["uv", "pip", "install", "--python", py38, "chemprop==1.7.0", "rdkit-pypi", "pandas", "numpy"], check=True)
    exe = find_chemprop_fingerprint()
    if not exe:
        raise RuntimeError("chemprop_fingerprint still not found after creating Python 3.8 env.")
    return exe

if not lion_ckpt.exists() or not ref_extra.exists():
    mark_status("LiON", "failed", "missing LNPDB LiON checkpoint or all_data_extra_x.csv")
else:
    chemprop_fp = ensure_lion_chemprop()

    extra_cols = pd.read_csv(ref_extra, nrows=0).columns.tolist()
    lion_work = subset.copy().reset_index(drop=True)
    lion_work["IL_SMILES"] = lion_work["il_smiles"].astype(str)
    lion_work["Experiment_value"] = pd.to_numeric(lion_work["delivery"], errors="coerce").fillna(0.0)

    X = pd.DataFrame(0.0, index=lion_work.index, columns=extra_cols)
    direct_map = {
        "IL_to_nucleicacid_massratio": "il_to_nucleic_acid_mass_ratio",
        "IL_molratio": "il_mol_ratio",
        "HL_molratio": "helper_lipid_mol_ratio",
        "CHL_molratio": "cholesterol_mol_ratio",
        "PEG_molratio": "peg_lipid_mol_ratio",
        "Dose_ug_nucleicacid": "dose_ug_nucleicacid",
    }
    for out_col, in_col in direct_map.items():
        if out_col in X.columns and in_col in lion_work.columns:
            X[out_col] = pd.to_numeric(lion_work[in_col], errors="coerce").fillna(0.0)

    def norm_name(x):
        return str(x).strip().replace(" ", "_").replace("/", "_").replace("-", "").replace(".", "")

    for prefix, source_col in [
        ("HL_name_", "helper_lipid_name"),
        ("CHL_name_", "cholesterol_name"),
        ("PEG_name_", "peg_lipid_name"),
        ("Model_type_", "biological_model"),
        ("Model_target_", "biological_target"),
        ("Route_of_administration_", "route_of_administration"),
        ("Cargo_", "cargo_type"),
        ("Cargo_type_", "cargo_type"),
    ]:
        if source_col not in lion_work.columns:
            continue
        source_norm = lion_work[source_col].map(norm_name)
        for col in [c for c in X.columns if c.startswith(prefix)]:
            X[col] = (source_norm == col[len(prefix):]).astype(float)

    for col in ["Route_of_administration_in_vitro", "Model_target_in_vitro"]:
        if col in X.columns:
            X[col] = 1.0

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    lion_tmp = ENCODED_DIR / "lion_input"
    lion_tmp.mkdir(parents=True, exist_ok=True)
    test_path = lion_tmp / "lnphub_lion_test.csv"
    features_path = lion_tmp / "lnphub_lion_extra_x.csv"
    raw_fp_path = lion_tmp / "LiON_raw_fingerprints.csv"

    lion_work[["IL_SMILES", "Experiment_value"]].to_csv(test_path, index=False)
    X.to_csv(features_path, index=False)

    print("LiON rows:", len(lion_work))
    print("LiON feature columns:", X.shape[1])
    print("Running chemprop_fingerprint...")

    res = shell([
        chemprop_fp,
        "--checkpoint_dir", str(lion_ckpt),
        "--test_path", str(test_path),
        "--features_path", str(features_path),
        "--preds_path", str(raw_fp_path),
    ])

    if res.returncode != 0 or not raw_fp_path.exists():
        mark_status("LiON", "failed", f"chemprop_fingerprint failed with code {res.returncode}")
    else:
        raw = pd.read_csv(raw_fp_path)
        vec_cols = [c for c in raw.columns if c not in {"smiles", "IL_SMILES", "Experiment_value"}]
        raw_vec = raw[vec_cols].apply(pd.to_numeric, errors="coerce").astype(np.float32)
        out_df = pd.concat([ids.reset_index(drop=True), raw_vec.reset_index(drop=True).add_prefix("lion_")], axis=1)
        out_path = ENCODED_DIR / "LiON_encoder_vectors.parquet"
        out_df.to_parquet(out_path, index=False)
        mark_status("LiON", "encoded", "completed", out_path)
        print(f"Saved LiON: {out_path}")
        print("Shape:", out_df.shape)
        display(out_df.head())

In [ ]:
# Final repo-based encoder status + quick QC
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

status = pd.read_csv(ENCODED_DIR / "encoder_status.csv") if (ENCODED_DIR / "encoder_status.csv").exists() else pd.DataFrame()
display(status)

qc_rows = []
for p in sorted(ENCODED_DIR.glob("*_encoder_vectors.parquet")):
    dfp = pd.read_parquet(p)
    id_like = {"record_id", "study_id", "lnp_id", "il_id", "il_smiles", "canonical_il_smiles", "delivery"}
    vec_cols = [c for c in dfp.columns if c not in id_like]
    X = dfp[vec_cols].apply(pd.to_numeric, errors="coerce").to_numpy()
    qc_rows.append({
        "file": p.name,
        "rows": dfp.shape[0],
        "vector_dim": len(vec_cols),
        "finite_fraction": float(np.isfinite(X).mean()) if X.size else np.nan,
        "nan_fraction": float(np.isnan(X).mean()) if X.size else np.nan,
        "all_zero": bool(np.nan_to_num(X, nan=0).sum() == 0) if X.size else True,
        "zero_variance_dims": int((np.nanstd(X, axis=0) < 1e-12).sum()) if X.size else 0,
    })

qc = pd.DataFrame(qc_rows)
display(qc)
qc.to_csv(ENCODED_DIR / "repo_based_encoder_vector_qc.csv", index=False)

print("Encoded vector files:")
for p in sorted(ENCODED_DIR.glob("*_encoder_vectors.parquet")):
    print("-", p)

## In-vivo expression oracle subset and encodings

This section extracts the in-vivo expression subset with more than 50 records per study, then encodes the ionizable-lipid SMILES using the same encoder workflow. All encoded outputs are saved under `/content/notebooks/encoded_in_vivo/`.

In [ ]:
# Cell 23 - Extract in-vivo expression oracle subset (>50 rows per study)
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

CSV_PATH = Path("/content/lnphub_master_v2.0.0.csv")
OUTDIR = Path("/content/notebooks/oracle_subsets")
ENCODED_DIR = Path("/content/notebooks/encoded_in_vivo")
OUTDIR.mkdir(parents=True, exist_ok=True)
ENCODED_DIR.mkdir(parents=True, exist_ok=True)

IN_VIVO_SUBSET_PATH = OUTDIR / "lnphub_oracle_in_vivo_expression_gt50.csv"
IN_VIVO_STUDY_SUMMARY_PATH = OUTDIR / "lnphub_oracle_in_vivo_expression_gt50_study_summary.csv"
IN_VIVO_QC_PATH = OUTDIR / "lnphub_oracle_in_vivo_expression_gt50_qc.csv"

required = ["record_id", "study_id", "experimental_setting", "delivery_metric", "delivery", "il_smiles"]
raw = pd.read_csv(CSV_PATH, low_memory=False)
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required column(s): {missing}")

work = raw.copy()
work["delivery_numeric"] = pd.to_numeric(work["delivery"], errors="coerce")

candidate = work[
    work["experimental_setting"].astype(str).str.strip().str.lower().eq("in_vivo") &
    work["delivery_metric"].astype(str).str.strip().str.lower().eq("expression") &
    work["delivery_numeric"].notna()
].copy()

study_counts = candidate.groupby("study_id", dropna=False).size().rename("n_rows").reset_index()
eligible_studies = study_counts.loc[study_counts["n_rows"] > 50, "study_id"]
subset_in_vivo = candidate[candidate["study_id"].isin(eligible_studies)].copy()
subset_in_vivo["delivery"] = subset_in_vivo["delivery_numeric"]
subset_in_vivo = subset_in_vivo.drop(columns=["delivery_numeric"])

study_summary_in_vivo = (
    subset_in_vivo.groupby("study_id", dropna=False)
    .agg(
        rows=("record_id", "size"),
        unique_lnps=("lnp_id", "nunique") if "lnp_id" in subset_in_vivo.columns else ("record_id", "size"),
        unique_il_smiles=("il_smiles", "nunique"),
        delivery_min=("delivery", "min"),
        delivery_median=("delivery", "median"),
        delivery_max=("delivery", "max"),
    )
    .sort_values("rows", ascending=False)
    .reset_index()
)

qc_in_vivo = pd.DataFrame([{
    "input_rows": len(raw),
    "candidate_in_vivo_expression_rows": len(candidate),
    "eligible_studies_gt50": int(eligible_studies.nunique()),
    "final_rows": len(subset_in_vivo),
    "final_unique_studies": subset_in_vivo["study_id"].nunique(),
    "final_unique_lnps": subset_in_vivo["lnp_id"].nunique() if "lnp_id" in subset_in_vivo.columns else np.nan,
    "final_unique_il_smiles": subset_in_vivo["il_smiles"].nunique(),
}])

subset_in_vivo.to_csv(IN_VIVO_SUBSET_PATH, index=False)
study_summary_in_vivo.to_csv(IN_VIVO_STUDY_SUMMARY_PATH, index=False)
qc_in_vivo.to_csv(IN_VIVO_QC_PATH, index=False)

print(f"Saved subset: {IN_VIVO_SUBSET_PATH}")
print(f"Saved study summary: {IN_VIVO_STUDY_SUMMARY_PATH}")
print(f"Saved QC: {IN_VIVO_QC_PATH}")
display(qc_in_vivo)
display(study_summary_in_vivo)

In [ ]:
# Cell 24 - Prepare in-vivo SMILES and identifiers for encoder outputs
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from rdkit import Chem

ENCODED_DIR = Path("/content/notebooks/encoded_in_vivo")
ENCODED_DIR.mkdir(parents=True, exist_ok=True)

subset = pd.read_csv(IN_VIVO_SUBSET_PATH, low_memory=False)
SMILES_COL = "il_smiles"

id_cols = [c for c in ["record_id", "study_id", "lnp_id", "il_id", "il_smiles", "delivery"] if c in subset.columns]
ids = subset[id_cols].copy().reset_index(drop=True)
if "record_id" not in ids.columns:
    ids.insert(0, "record_id", np.arange(len(ids)))

if SMILES_COL not in subset.columns:
    raise ValueError(f"Missing {SMILES_COL}")

def canonical_smiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else np.nan

ids["canonical_il_smiles"] = subset[SMILES_COL].map(canonical_smiles)
valid_ids = ids.dropna(subset=["canonical_il_smiles"]).copy()
unique_smiles = (
    valid_ids[["canonical_il_smiles"]]
    .drop_duplicates()
    .sort_values("canonical_il_smiles")
    .reset_index(drop=True)
)
unique_smiles["smiles_index"] = np.arange(len(unique_smiles))

valid_ids = valid_ids.merge(unique_smiles, on="canonical_il_smiles", how="left")

print(f"In-vivo rows: {len(ids):,}")
print(f"Valid RDKit rows: {len(valid_ids):,}")
print(f"Unique valid ionizable-lipid SMILES: {len(unique_smiles):,}")
display(valid_ids.head())

In [ ]:
# Cell 25 - Common in-vivo encoder save/QC helpers
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

ENCODED_DIR = Path("/content/notebooks/encoded_in_vivo")
ENCODED_DIR.mkdir(parents=True, exist_ok=True)
encoder_status = []

ID_LIKE_COLS = {"record_id", "study_id", "lnp_id", "il_id", "il_smiles", "canonical_il_smiles", "delivery", "smiles_index"}

def save_encoder_vectors(encoder_name, vectors, feature_prefix):
    vectors = np.asarray(vectors, dtype=np.float32)
    if vectors.shape[0] != len(unique_smiles):
        raise ValueError(f"{encoder_name}: vector rows {vectors.shape[0]} != unique SMILES {len(unique_smiles)}")

    vec_cols = [f"{feature_prefix}_{i:04d}" for i in range(vectors.shape[1])]
    unique_vecs = pd.concat(
        [unique_smiles.copy(), pd.DataFrame(vectors, columns=vec_cols)],
        axis=1,
    )

    out = valid_ids.merge(unique_vecs, on=["canonical_il_smiles", "smiles_index"], how="left")
    out_path = ENCODED_DIR / f"{encoder_name}_encoder_vectors.parquet"
    out.to_parquet(out_path, index=False)

    qc = pd.DataFrame([{
        "encoder": encoder_name,
        "rows": out.shape[0],
        "vector_dim": len(vec_cols),
        "finite_fraction": float(np.isfinite(out[vec_cols].to_numpy()).mean()),
        "nan_fraction": float(np.isnan(out[vec_cols].to_numpy()).mean()),
        "all_zero": bool(np.nan_to_num(out[vec_cols].to_numpy(), nan=0).sum() == 0),
        "output_file": str(out_path),
    }])
    qc.to_csv(ENCODED_DIR / f"{encoder_name}_qc.csv", index=False)

    print(f"Saved {encoder_name}: {out_path}")
    print(f"Shape: {out.shape[0]:,} rows x {out.shape[1]:,} columns")
    display(qc)
    return out_path

def mark_status(encoder, status, note="", output_file=None):
    row = {"encoder": encoder, "status": status, "note": note, "output_file": str(output_file) if output_file else ""}
    encoder_status.append(row)
    pd.DataFrame(encoder_status).to_csv(ENCODED_DIR / "encoder_status.csv", index=False)
    print(f"[{encoder}] {status}: {note}")

print(f"In-vivo encoded outputs will be written to: {ENCODED_DIR}")

In [ ]:
# Cell 26 - Install/check encoder dependencies for the current Colab runtime
import sys, subprocess, importlib.util
import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

packages = [
    "transformers",
    "accelerate",
    "safetensors",
    "rdkit-pypi",
    "unimol-tools",
    "torch_geometric",
    "pyarrow",
    "tqdm",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=False)

In [ ]:
# Cell 27 - Encode in-vivo subset with ChemBERTa-MTR and MoLFormer
import torch
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch device: {DEVICE}")

@torch.inference_mode()
def encode_hf_smiles(model_id, encoder_name, prefix, batch_size=32, max_length=256, trust_remote_code=True):
    print(f"\n=== {encoder_name} ===")
    print(f"Loading tokenizer/model: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    try:
        model = AutoModel.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    except Exception as e:
        print(f"AutoModel failed, trying AutoModelForSequenceClassification: {e}")
        model = AutoModelForSequenceClassification.from_pretrained(model_id, trust_remote_code=trust_remote_code)

    model.to(DEVICE)
    model.eval()

    smiles = unique_smiles["canonical_il_smiles"].astype(str).tolist()
    chunks = []
    for start in tqdm(range(0, len(smiles), batch_size), desc=f"Encoding {encoder_name}"):
        batch = smiles[start:start + batch_size]
        tok = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        out = model(**tok, output_hidden_states=True, return_dict=True)

        hidden = getattr(out, "last_hidden_state", None)
        if hidden is None:
            hidden = out.hidden_states[-1]

        mask = tok.get("attention_mask", torch.ones(hidden.shape[:2], device=hidden.device)).unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        chunks.append(pooled.detach().cpu().numpy().astype(np.float32))

    vectors = np.vstack(chunks)
    return save_encoder_vectors(encoder_name, vectors, prefix)

hf_jobs = [
    ("DeepChem/ChemBERTa-77M-MTR", "ChemBERTa_MTR", "chemberta_mtr", 32, 256, False),
    ("ibm-research/MoLFormer-XL-both-10pct", "MoLFormer", "molformer", 16, 256, True),
]

for model_id, encoder_name, prefix, batch_size, max_length, trust_remote_code in hf_jobs:
    try:
        out = encode_hf_smiles(model_id, encoder_name, prefix, batch_size, max_length, trust_remote_code)
        mark_status(encoder_name, "encoded", "completed", out)
    except Exception as e:
        mark_status(encoder_name, "failed", repr(e))

In [ ]:
# Cell 28 - Encode in-vivo subset with Uni-Mol
import numpy as np
from tqdm.auto import tqdm

try:
    from unimol_tools import UniMolRepr

    def extract_unimol_cls_repr(rep):
        if isinstance(rep, dict):
            if "cls_repr" in rep:
                return np.asarray(rep["cls_repr"], dtype=np.float32)
            if "molecular_repr" in rep:
                return np.asarray(rep["molecular_repr"], dtype=np.float32)

        if isinstance(rep, list):
            if len(rep) and isinstance(rep[0], dict):
                key = "cls_repr" if "cls_repr" in rep[0] else "molecular_repr"
                return np.vstack([np.asarray(x[key], dtype=np.float32) for x in rep])
            return np.asarray(rep, dtype=np.float32)

        arr = np.asarray(rep, dtype=np.float32)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        return arr

    print("\n=== Uni-Mol ===")
    smiles = unique_smiles["canonical_il_smiles"].astype(str).tolist()
    clf = UniMolRepr(data_type="molecule", remove_hs=False)

    chunks = []
    BATCH_SIZE = 64
    for start in tqdm(range(0, len(smiles), BATCH_SIZE), desc="Encoding Uni-Mol"):
        batch = smiles[start:start + BATCH_SIZE]
        rep = clf.get_repr(batch, return_atomic_reprs=False)
        chunks.append(extract_unimol_cls_repr(rep))

    vectors = np.vstack(chunks).astype(np.float32)
    out = save_encoder_vectors("Uni_Mol", vectors, "unimol")
    mark_status("Uni_Mol", "encoded", "completed", out)
except Exception as e:
    mark_status("Uni_Mol", "failed", repr(e))


In [ ]:
# Cell 29 - Reuse/clone official repos and verify repo-based checkpoints
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

REPO_DIR = Path("/content/notebooks/encoder_repos")
REPO_DIR.mkdir(parents=True, exist_ok=True)

def clone_if_needed(name, url):
    dest = REPO_DIR / name
    if dest.exists():
        print(f"{name}: repo already present -> {dest}")
        return dest
    print(f"{name}: cloning {url}")
    subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=False)
    return dest

AGILE_DIR = clone_if_needed("AGILE", "https://github.com/bowang-lab/AGILE.git")
MOLCLR_DIR = clone_if_needed("MolCLR", "https://github.com/yuyangw/MolCLR.git")
KPGT_DIR = clone_if_needed("KPGT", "https://github.com/lihan97/kpgt.git")
LNPDB_DIR = clone_if_needed("LNPDB", "https://github.com/evancollins1/LNPDB.git")

ckpt_checks = pd.DataFrame([
    {"encoder": "AGILE", "path": AGILE_DIR / "ckpt" / "pretrained_agile_60k" / "checkpoints" / "model.pth"},
    {"encoder": "MolCLR_GIN", "path": MOLCLR_DIR / "ckpt" / "pretrained_gin" / "checkpoints" / "model.pth"},
    {"encoder": "KPGT", "path": KPGT_DIR / "models" / "pretrained" / "base" / "base.pth"},
    {"encoder": "LiON", "path": LNPDB_DIR / "data" / "LNPDB_for_LiON" / "single_split" / "trained_model_checkpoints"},
])
ckpt_checks["exists"] = ckpt_checks["path"].map(lambda p: Path(p).exists())
ckpt_checks["size_MB"] = ckpt_checks["path"].map(lambda p: round(Path(p).stat().st_size / 1e6, 3) if Path(p).is_file() else np.nan)
ckpt_checks["path"] = ckpt_checks["path"].astype(str)
ckpt_checks.to_csv(ENCODED_DIR / "repo_based_encoder_access_check.csv", index=False)
display(ckpt_checks)

In [ ]:
# Cell 30 - Encode in-vivo subset with AGILE and MolCLR-GIN
import sys, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from rdkit import Chem
from torch_geometric.data import Data, Batch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

ATOM_LIST = list(range(1, 119))
CHIRALITY_LIST = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
]
BOND_LIST = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]
BONDDIR_LIST = [
    Chem.rdchem.BondDir.NONE,
    Chem.rdchem.BondDir.ENDUPRIGHT,
    Chem.rdchem.BondDir.ENDDOWNRIGHT,
]

def load_class_from_file(path, class_name, module_name):
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return getattr(mod, class_name)

def smiles_to_pyg_graph(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    mol = Chem.AddHs(mol)

    atoms = []
    for atom in mol.GetAtoms():
        atomic_num = atom.GetAtomicNum()
        if atomic_num not in ATOM_LIST:
            return None
        chiral = atom.GetChiralTag()
        chirality_idx = CHIRALITY_LIST.index(chiral) if chiral in CHIRALITY_LIST else 0
        atoms.append([ATOM_LIST.index(atomic_num), chirality_idx])

    row, col, edge_feat = [], [], []
    for bond in mol.GetBonds():
        bt, bd = bond.GetBondType(), bond.GetBondDir()
        btype = BOND_LIST.index(bt) if bt in BOND_LIST else 0
        bdir = BONDDIR_LIST.index(bd) if bd in BONDDIR_LIST else 0
        a, b = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        row += [a, b]
        col += [b, a]
        edge_feat += [[btype, bdir], [btype, bdir]]

    return Data(
        x=torch.tensor(atoms, dtype=torch.long),
        edge_index=torch.tensor([row, col], dtype=torch.long),
        edge_attr=torch.tensor(edge_feat, dtype=torch.long),
        y=torch.zeros((1, 1), dtype=torch.float),
    )

def load_state_dict_safely(path):
    obj = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(obj, dict) and "state_dict" in obj:
        obj = obj["state_dict"]
    if isinstance(obj, dict) and "model_state_dict" in obj:
        obj = obj["model_state_dict"]
    return {k.replace("module.", ""): v for k, v in obj.items()}

@torch.inference_mode()
def encode_gnn_checkpoint(encoder_name, model_class, ckpt_path, prefix, batch_size=128):
    print(f"\n=== {encoder_name} ===")
    ckpt_path = Path(ckpt_path)
    print("Checkpoint:", ckpt_path)

    if not ckpt_path.exists() or ckpt_path.stat().st_size < 100000:
        mark_status(encoder_name, "failed", f"missing or incomplete checkpoint: {ckpt_path}")
        return None

    model = model_class(num_layer=5, emb_dim=300, feat_dim=512, drop_ratio=0, pool="mean").to(DEVICE)
    msg = model.load_state_dict(load_state_dict_safely(ckpt_path), strict=False)
    print("Missing keys:", len(msg.missing_keys), "Unexpected keys:", len(msg.unexpected_keys))
    model.eval()

    smiles = unique_smiles["canonical_il_smiles"].astype(str).tolist()
    vectors = np.full((len(smiles), 512), np.nan, dtype=np.float32)
    failed, graph_items = [], []

    for i, smi in enumerate(tqdm(smiles, desc=f"Building graphs for {encoder_name}")):
        g = smiles_to_pyg_graph(smi)
        if g is None:
            failed.append({"row": i, "canonical_il_smiles": smi})
        else:
            graph_items.append((i, g))

    print(f"Graphs built: {len(graph_items):,}; failed: {len(failed):,}")
    if failed:
        pd.DataFrame(failed).to_csv(ENCODED_DIR / f"{prefix}_failed_smiles.csv", index=False)

    for start in tqdm(range(0, len(graph_items), batch_size), desc=f"Encoding {encoder_name}"):
        batch_items = graph_items[start:start + batch_size]
        idxs = [i for i, _ in batch_items]
        batch = Batch.from_data_list([g for _, g in batch_items]).to(DEVICE)
        h, _ = model(batch)
        vectors[idxs, :] = h.detach().cpu().numpy().astype(np.float32)

    out = save_encoder_vectors(encoder_name, vectors, prefix)
    mark_status(encoder_name, "encoded", f"completed; failed_graphs={len(failed)}", out)
    return out

AGILE = load_class_from_file(AGILE_DIR / "models" / "agile_pretrain.py", "AGILE", "agile_pretrain_in_vivo")
GINet = load_class_from_file(MOLCLR_DIR / "models" / "ginet_molclr.py", "GINet", "molclr_ginet_in_vivo")

encode_gnn_checkpoint(
    "AGILE",
    AGILE,
    AGILE_DIR / "ckpt" / "pretrained_agile_60k" / "checkpoints" / "model.pth",
    "agile",
)

encode_gnn_checkpoint(
    "MolCLR_GIN",
    GINet,
    MOLCLR_DIR / "ckpt" / "pretrained_gin" / "checkpoints" / "model.pth",
    "molclr_gin",
)

In [ ]:
# Cell 31 - Encode in-vivo subset with KPGT if the official base.pth checkpoint is available
import shutil, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

print("\n=== KPGT ===")
kpgt_root = Path(KPGT_DIR)
kpgt_data_root = kpgt_root / "datasets"
kpgt_dataset = "lnphub_in_vivo_expression"
kpgt_dataset_dir = kpgt_data_root / kpgt_dataset
kpgt_dataset_dir.mkdir(parents=True, exist_ok=True)

kpgt_input = unique_smiles.rename(columns={"canonical_il_smiles": "smiles"}).copy()
kpgt_input["dummy"] = 0.0
kpgt_csv = kpgt_dataset_dir / f"{kpgt_dataset}.csv"
kpgt_input.to_csv(kpgt_csv, index=False)
print(f"KPGT input saved: {kpgt_csv}")

kpgt_model_path = kpgt_root / "models" / "pretrained" / "base" / "base.pth"
for candidate in [Path("/content/base.pth"), Path("/content/notebooks/base.pth"), ENCODED_DIR / "base.pth"]:
    if candidate.exists() and not kpgt_model_path.exists():
        kpgt_model_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(candidate, kpgt_model_path)
        print(f"Copied KPGT checkpoint from {candidate} -> {kpgt_model_path}")

if not kpgt_model_path.exists():
    mark_status(
        "KPGT",
        "failed",
        "missing checkpoint base.pth; place it at /content/notebooks/encoder_repos/KPGT/models/pretrained/base/base.pth",
    )
    display(pd.DataFrame([{
        "encoder": "KPGT",
        "status": "missing_checkpoint",
        "needed_file": str(kpgt_model_path),
        "note": "Download the official pretrained KPGT base.pth from the Figshare link in the KPGT README, then rerun this cell.",
    }]))
else:
    print("Preprocessing KPGT graphs, fingerprints, and descriptors...")
    subprocess.run(
        [sys.executable, "scripts/preprocess_downstream_dataset.py", "--data_path", str(kpgt_data_root), "--dataset", kpgt_dataset, "--n_jobs", "2"],
        cwd=str(kpgt_root),
        check=True,
    )

    print("Extracting KPGT latent features...")
    subprocess.run(
        [sys.executable, "extract_features.py", "--config", "base", "--model_path", str(kpgt_model_path), "--data_path", str(kpgt_data_root), "--dataset", kpgt_dataset],
        cwd=str(kpgt_root / "scripts"),
        check=True,
    )

    npz_path = kpgt_dataset_dir / "kpgt_base.npz"
    vectors = np.load(npz_path)["fps"].astype(np.float32)
    out = save_encoder_vectors("KPGT", vectors, "kpgt")
    mark_status("KPGT", "encoded", "completed", out)

In [ ]:
# Cell 32 - Encode in-vivo subset with LiON using the LNPDB Chemprop checkpoint
# LiON is formulation-level: it uses IL_SMILES plus formulation/context features aligned to LNPDB extra_x.
# The local Python 3.8 environment is used because chemprop==1.7.0 is not compatible with newer Colab Python.

import os, sys, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

print("\n=== LiON ===")
lion_dir = Path(LNPDB_DIR) / "data" / "LNPDB_for_LiON"
lion_ckpt = lion_dir / "single_split" / "trained_model_checkpoints"
ref_extra = lion_dir / "all_data_extra_x.csv"

LION_VENV = Path("/content/notebooks/lion_py38")

def shell(cmd, check=False):
    print("$", " ".join(map(str, cmd)))
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.stdout:
        print(res.stdout[-3500:])
    if res.stderr:
        print(res.stderr[-3500:])
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed with code {res.returncode}: {' '.join(map(str, cmd))}")
    return res

def ensure_uv():
    if shutil.which("uv"):
        return "uv"
    shell([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    return shutil.which("uv") or "uv"

def find_chemprop_fingerprint():
    exe = shutil.which("chemprop_fingerprint")
    if exe:
        return exe
    venv_exe = LION_VENV / "bin" / "chemprop_fingerprint"
    return str(venv_exe) if venv_exe.exists() else None

def ensure_lion_chemprop():
    uv = ensure_uv()
    py38 = LION_VENV / "bin" / "python"

    if not py38.exists():
        print("Creating Python 3.8 env for LiON/Chemprop...")
        shell([uv, "python", "install", "3.8"], check=True)
        shell([uv, "venv", str(LION_VENV), "--python", "3.8"], check=True)

    shell([uv, "pip", "install", "--python", str(py38), "setuptools<70", "chemprop==1.7.0", "rdkit-pypi", "pandas", "numpy"], check=True)
    exe = find_chemprop_fingerprint()
    if not exe:
        raise RuntimeError("chemprop_fingerprint not found after preparing Python 3.8 env.")
    print("Using chemprop_fingerprint:", exe)
    return exe

if not lion_ckpt.exists() or not ref_extra.exists():
    mark_status("LiON", "failed", "missing LNPDB LiON checkpoint or all_data_extra_x.csv")
else:
    chemprop_fp = ensure_lion_chemprop()

    extra_cols = pd.read_csv(ref_extra, nrows=0).columns.tolist()
    lion_work = subset.copy().reset_index(drop=True)
    lion_work["IL_SMILES"] = lion_work["il_smiles"].astype(str)
    lion_work["Experiment_value"] = pd.to_numeric(lion_work["delivery"], errors="coerce").fillna(0.0)

    X = pd.DataFrame(0.0, index=lion_work.index, columns=extra_cols)
    direct_map = {
        "IL_to_nucleicacid_massratio": "il_to_nucleic_acid_mass_ratio",
        "IL_molratio": "il_mol_ratio",
        "HL_molratio": "helper_lipid_mol_ratio",
        "CHL_molratio": "cholesterol_mol_ratio",
        "PEG_molratio": "peg_lipid_mol_ratio",
        "Dose_ug_nucleicacid": "dose_ug_nucleicacid",
    }
    for out_col, in_col in direct_map.items():
        if out_col in X.columns and in_col in lion_work.columns:
            X[out_col] = pd.to_numeric(lion_work[in_col], errors="coerce").fillna(0.0)

    def norm_name(x):
        return str(x).strip().replace(" ", "_").replace("/", "_").replace("-", "").replace(".", "")

    for prefix, source_col in [
        ("HL_name_", "helper_lipid_name"),
        ("CHL_name_", "cholesterol_name"),
        ("PEG_name_", "peg_lipid_name"),
        ("Model_type_", "biological_model"),
        ("Model_target_", "biological_target"),
        ("Route_of_administration_", "route_of_administration"),
        ("Cargo_", "cargo_type"),
        ("Cargo_type_", "cargo_type"),
    ]:
        if source_col not in lion_work.columns:
            continue
        source_norm = lion_work[source_col].map(norm_name)
        for col in [c for c in X.columns if c.startswith(prefix)]:
            X[col] = (source_norm == col[len(prefix):]).astype(float)

    for col in ["Route_of_administration_in_vivo", "Model_target_in_vivo"]:
        if col in X.columns:
            X[col] = 1.0

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    lion_tmp = ENCODED_DIR / "lion_input"
    lion_tmp.mkdir(parents=True, exist_ok=True)
    test_path = lion_tmp / "lnphub_lion_test.csv"
    features_path = lion_tmp / "lnphub_lion_extra_x.csv"
    raw_fp_path = lion_tmp / "LiON_raw_fingerprints.csv"

    lion_work[["IL_SMILES", "Experiment_value"]].to_csv(test_path, index=False)
    X.to_csv(features_path, index=False)

    print("LiON rows:", len(lion_work))
    print("LiON feature columns:", X.shape[1])
    print("Running chemprop_fingerprint...")

    res = shell([
        chemprop_fp,
        "--checkpoint_dir", str(lion_ckpt),
        "--test_path", str(test_path),
        "--features_path", str(features_path),
        "--preds_path", str(raw_fp_path),
    ])

    if res.returncode != 0 or not raw_fp_path.exists():
        mark_status("LiON", "failed", f"chemprop_fingerprint failed with code {res.returncode}")
    else:
        raw_fp = pd.read_csv(raw_fp_path)
        fp_cols = [c for c in raw_fp.columns if c not in {"smiles", "IL_SMILES", "Experiment_value", "preds"}]
        vectors = raw_fp[fp_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)

        if vectors.shape[0] != len(lion_work):
            mark_status("LiON", "failed", f"LiON row mismatch: {vectors.shape[0]} fingerprints for {len(lion_work)} rows")
        else:
            vec_cols = [f"lion_{i:04d}" for i in range(vectors.shape[1])]
            out = pd.concat([ids.reset_index(drop=True), pd.DataFrame(vectors, columns=vec_cols)], axis=1)
            out_path = ENCODED_DIR / "LiON_encoder_vectors.parquet"
            out.to_parquet(out_path, index=False)

            qc = pd.DataFrame([{
                "encoder": "LiON",
                "rows": out.shape[0],
                "vector_dim": len(vec_cols),
                "finite_fraction": float(np.isfinite(out[vec_cols].to_numpy()).mean()),
                "nan_fraction": float(np.isnan(out[vec_cols].to_numpy()).mean()),
                "all_zero": bool(np.nan_to_num(out[vec_cols].to_numpy(), nan=0).sum() == 0),
                "output_file": str(out_path),
            }])
            qc.to_csv(ENCODED_DIR / "LiON_qc.csv", index=False)
            display(qc)
            mark_status("LiON", "encoded", "completed", out_path)

In [ ]:
# Cell 33 - Final in-vivo encoder status and vector QC
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

ENCODED_DIR = Path("/content/notebooks/encoded_in_vivo")
status_path = ENCODED_DIR / "encoder_status.csv"
status = pd.read_csv(status_path) if status_path.exists() else pd.DataFrame()
print("Encoder status:")
display(status)

qc_rows = []
id_like = {"record_id", "study_id", "lnp_id", "il_id", "il_smiles", "canonical_il_smiles", "delivery", "smiles_index"}
for p in sorted(ENCODED_DIR.glob("*_encoder_vectors.parquet")):
    dfp = pd.read_parquet(p)
    vec_cols = [c for c in dfp.columns if c not in id_like]
    X = dfp[vec_cols].apply(pd.to_numeric, errors="coerce").to_numpy()
    qc_rows.append({
        "file": p.name,
        "rows": dfp.shape[0],
        "vector_dim": len(vec_cols),
        "finite_fraction": float(np.isfinite(X).mean()) if X.size else np.nan,
        "nan_fraction": float(np.isnan(X).mean()) if X.size else np.nan,
        "all_zero": bool(np.nan_to_num(X, nan=0).sum() == 0) if X.size else True,
        "zero_variance_dims": int((np.nanstd(X, axis=0) < 1e-12).sum()) if X.size else 0,
    })

qc = pd.DataFrame(qc_rows)
qc.to_csv(ENCODED_DIR / "encoder_vector_qc.csv", index=False)
print("Vector QC:")
display(qc)

print("Encoded vector files:")
for p in sorted(ENCODED_DIR.glob("*_encoder_vectors.parquet")):
    print("-", p)

print("\nTo download the full in-vivo encoded folder, run:")
print("from google.colab import files; import shutil")
print("shutil.make_archive('/content/encoded_in_vivo', 'zip', '/content/notebooks/encoded_in_vivo')")
print("files.download('/content/encoded_in_vivo.zip')")